# Notebook 2: Cross-Repository Evaluation Overview

**Goal:** Describe the study population and overall evaluation scope before ranking any approach.

**Inputs:** `evaluation_attempts.csv`, `evaluation_overview.json`

**RQs addressed:** RQ1 (scope), RQ2 (scope)

In [ ]:
import json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')

DATA_DIR = Path('../output')
attempts  = pd.read_csv(DATA_DIR / 'evaluation_attempts.csv')

overview_path = DATA_DIR / 'evaluation_overview.json'
overview = json.loads(overview_path.read_text()) if overview_path.exists() else {}

## 1. What Was Evaluated?

In [ ]:
scope = overview.get('scope', {})
scope_df = pd.DataFrame(list(scope.items()), columns=['metric', 'value'])
display(scope_df)

# Headline cards
for k, v in scope.items():
    print(f'  {k}: {v}')

## 2. How Much Was Generated?

In [ ]:
gen = overview.get('generated_tests', {})
print('Generated test counts:')
for k, v in gen.items():
    print(f'  {k}: {v}')

if 'generated_test_count' in attempts.columns and 'lane' in attempts.columns:
    fig, ax = plt.subplots(figsize=(8, 4))
    attempts.groupby('lane')['generated_test_count'].sum().plot(kind='bar', ax=ax)
    ax.set_title('Total generated tests by lane')
    ax.set_ylabel('Test count')
    plt.tight_layout()
    plt.show()

## 3. What Happened Overall?

In [ ]:
outcomes = overview.get('outcomes', {})
print('Outcome counts:')
for k, v in outcomes.items():
    print(f'  {k}: {v}')

if 'lane' in attempts.columns and 'validated_success' in attempts.columns:
    summary = attempts.groupby('lane')['validated_success'].agg(['sum', 'count', 'mean'])
    summary.columns = ['successes', 'total', 'rate']
    display(summary)

## 4. How Much Did Metrics Move?

In [ ]:
movement = overview.get('metric_movement', {})
print('Metric movement:')
for k, v in movement.items():
    print(f'  {k}: {v}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
if 'coverage_delta' in attempts.columns:
    attempts['coverage_delta'].dropna().hist(bins=30, ax=axes[0])
    axes[0].axvline(0, color='red', linestyle='--')
    axes[0].set_title('Coverage delta distribution')
if 'mutation_score_delta' in attempts.columns:
    attempts['mutation_score_delta'].dropna().hist(bins=30, ax=axes[1])
    axes[1].axvline(0, color='red', linestyle='--')
    axes[1].set_title('Mutation score delta distribution')
plt.tight_layout()
plt.show()

## 5. What Did It Cost?

In [ ]:
cost = overview.get('cost', {})
print('Cost / runtime:')
for k, v in cost.items():
    print(f'  {k}: {v}')

if 'duration_seconds' in attempts.columns and 'lane' in attempts.columns:
    display(attempts.groupby('lane')['duration_seconds'].agg(['median', 'sum', 'count']))

## 6. How Complete Is the Data?

In [ ]:
completeness = overview.get('data_completeness', {})
print('Missingness:')
for k, v in completeness.items():
    print(f'  {k}: {v}')

# Heatmap of null % by lane × column
key_cols = ['coverage_before', 'coverage_after', 'mutation_score_before',
            'mutation_score_after', 'total_tokens', 'duration_seconds']
present = [c for c in key_cols if c in attempts.columns]
if present and 'lane' in attempts.columns:
    null_pct = attempts.groupby('lane')[present].apply(lambda g: g.isna().mean() * 100)
    fig, ax = plt.subplots(figsize=(10, 3))
    sns.heatmap(null_pct, annot=True, fmt='.1f', cmap='Reds', ax=ax, cbar_kws={'label': '% missing'})
    ax.set_title('Data completeness by lane (% missing)')
    plt.tight_layout()
    plt.show()